In [2]:
import platform
import subprocess
import re
import time
import threading

# Credits: https://stackoverflow.com/a/65447493/15488797
class ThreadWithResult(threading.Thread):
    def __init__(self, group=None, target=None, name=None, args=(), kwargs={}, *, daemon=None):
        def function():
            self.result = target(*args, **kwargs)
        super().__init__(group=group, target=function, name=name, daemon=daemon)

# Returns latency from pinging a server with threading
def ping_threading(host, id=0):
    param = '-n' if platform.system().lower()=='windows' else '-c'
    p = subprocess.Popen(["ping", param, '1', host], stdout = subprocess.PIPE, stderr = subprocess.STDOUT)
    latency = re.search("time=([0-9]+.[0-9]+) ms", str(p.communicate()[0])).group(1)
    return float(latency)

def main(host, repeat=10):
    threads = []
    start = time.time()
    for i in range(1, repeat+1):
        x = ThreadWithResult(target=ping_threading, args=(host, i, ))
        x.start()
        threads.append(x)
    for x in threads:
        x.join()
    end = (time.time() - start)*1000
    total = sum([thread.result for thread in threads])
    results = {
                'latency in milliseconds asynchronous': end, 
                'latency in milliseconds synchronous': total, 
              }
    return results

main("www.google.com", repeat=10)

{'latency in milliseconds asynchronous': 101.93800926208496,
 'latency in milliseconds synchronous': 177.495}

In [28]:
class Solution:
    def exist(self, board, word):
        for i in range(len(board)):
            for j in range(len(board[0])):
                if self.dfs(board, word, i, j):
                    return True
        return False
    def dfs(self, board, word, i, j):
        if len(word) == 0:
            return True
        if not (0 <= i < len(board)) or not (0 <= j < len(board[0])) or (board[i][j] != word[0]):
            return False
        save, board[i][j] = board[i][j], '#'
        for new_i, new_j in [(i+1, j), (i-1, j), (i, j-1), (i, j+1)]:
            result = self.dfs(board, word[1:], new_i, new_j)
            if result:
                board[i][j] = save
                return True
        board[i][j] = save
        return False

In [29]:
s = Solution()
s.exist([["A"]], "A")

True

In [53]:
class Solution:
    def maxSubArray(self, nums):
        if len(nums) == 0: return 0
        prev = maxSum = nums[0]
        for num in nums[1:]:
            prev = max(prev + num, num)
            maxSum = max(prev, maxSum)
        return maxSum

In [55]:
s = Solution()
s.maxSubArray([5,4,-1,7,8])

23

In [113]:
import urllib3.request, json 
import urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
from tqdm import tqdm
import numpy as np
#!pip install editdistance
import editdistance
import os
from pathlib import Path
import requests
from collections import Counter
import re

In [2]:
!curl -O http://api.alquran.cloud/v1/quran/ar.alafasy

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 6051k  100 6051k    0     0   553k      0  0:00:10  0:00:10 --:--:-- 1640k


In [40]:
!curl -O https://everyayah.com/data/recitations.js

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 10907  100 10907    0     0   2663      0  0:00:04  0:00:04 --:--:--  2663


In [20]:
identifier = "ar.alafasy"
info_json = json.load(open(identifier+'.json', 'r'))
Path(identifier).mkdir(parents=True, exist_ok=True)

In [149]:
recitations_json = json.load(open('recitations.json', 'r'))

In [150]:
recitations_json['1']

{'subfolder': 'Abdul_Basit_Murattal_64kbps',
 'name': 'Abdul Basit Murattal',
 'bitrate': '64kbps'}

In [151]:
conversion = {'Ghamadi': '', 'Mohammad_al_Tablaway_64kbps': '', 'Mohammad_al_Tablaway_128kbps': '', 'Mustafa_Ismail_48kbps': '', 'English/Sahih_Intnl_Ibrahim_Walk_192kbps': 'en', 'MultiLanguage/Basfar_Walk_192kbps': '', 'translations/Makarem_Kabiri_16Kbps': '', 'translations/Fooladvand_Hedayatfar_40Kbps': 'fa', 'translations/azerbaijani/balayev': 'az', 'Salaah_AbdulRahman_Bukhatir_128kbps': '', 'Muhsin_Al_Qasim_192kbps': '', 'Abdullaah_3awwaad_Al-Juhaynee_128kbps': '', 'Salah_Al_Budair_128kbps': '', 'Abdullah_Matroud_128kbps': '', 'Ahmed_Neana_128kbps': '', 'Muhammad_AbdulKareem_128kbps': '', 'khalefa_al_tunaiji_64kbps': '', 'mahmoud_ali_al_banna_32kbps': '', 'warsh/warsh_ibrahim_aldosary_128kbps': '', 'warsh/warsh_yassin_al_jazaery_64kbps': '', 'warsh/warsh_Abdul_Basit_128kbps': '', 'translations/urdu_shamshad_ali_khan_46kbps': 'ur', 'Karim_Mansoori_40kbps': '', 'Khaalid_Abdullaah_al-Qahtaanee_192kbps': '', 'Yasser_Ad-Dussary_128kbps': '', 'Nasser_Alqatami_128kbps': '', 'Ali_Hajjaj_AlSuesy_128kbps': '', 'Sahl_Yassin_128kbps': '', 'translations/besim_korkut_ajet_po_ajet': '', 'aziz_alili_128kbps': '', 'Yaser_Salamah_128kbps': '', 'Akram_AlAlaqimy_128kbps': '', 'Ali_Jaber_64kbps': '', 'Fares_Abbad_64kbps': '', 'translations/urdu_farhat_hashmi': 'ur'}

In [152]:
for key in recitations_json:
    if key == 'ayahCount': continue
    recitation = recitations_json[key]
    subfolder = recitation['subfolder']
    if subfolder in conversion:
        recitations_json[key]['language'] = conversion[subfolder]
    else:
        recitations_json[key]['language'] = 'ar'

In [153]:
recitations_json['1']

{'subfolder': 'Abdul_Basit_Murattal_64kbps',
 'name': 'Abdul Basit Murattal',
 'bitrate': '64kbps',
 'language': 'ar'}

In [154]:
names = Counter([recitations_json[item]['name'] for item in recitations_json if item != 'ayahCount'])

In [155]:
pattern = re.compile(r'(\d+)kbps')
for item in recitations_json:
    if item == 'ayahCount': continue
    name = recitations_json[item]['name']
    if names[name] == 1: continue
    subfolder = recitations_json[item]['subfolder']
    value = pattern.findall(subfolder)[0]
    recitations_json[item]['name'] += ' ({} kbps)'.format(value)

In [156]:
recitations_json['1']

{'subfolder': 'Abdul_Basit_Murattal_64kbps',
 'name': 'Abdul Basit Murattal (64 kbps)',
 'bitrate': '64kbps',
 'language': 'ar'}

In [86]:
[(key, recitations_json[key]['subfolder']) for key in recitations_json if key != 'ayahCount']

[('1', 'Abdul_Basit_Murattal_64kbps'),
 ('2', 'Abdul_Basit_Murattal_192kbps'),
 ('3', 'Abdul_Basit_Mujawwad_128kbps'),
 ('4', 'Abdullah_Basfar_32kbps'),
 ('5', 'Abdullah_Basfar_64kbps'),
 ('6', 'Abdullah_Basfar_192kbps'),
 ('7', 'Abdurrahmaan_As-Sudais_64kbps'),
 ('8', 'Abdurrahmaan_As-Sudais_192kbps'),
 ('9', 'AbdulSamad_64kbps_QuranExplorer.Com'),
 ('10', 'Abu_Bakr_Ash-Shaatree_64kbps'),
 ('11', 'Abu_Bakr_Ash-Shaatree_128kbps'),
 ('12', 'Ahmed_ibn_Ali_al-Ajamy_64kbps_QuranExplorer.Com'),
 ('13', 'Ahmed_ibn_Ali_al-Ajamy_128kbps_ketaballah.net'),
 ('14', 'Alafasy_64kbps'),
 ('15', 'Alafasy_128kbps'),
 ('16', 'Ghamadi_40kbps'),
 ('17', 'Hani_Rifai_64kbps'),
 ('18', 'Hani_Rifai_192kbps'),
 ('19', 'Husary_64kbps'),
 ('20', 'Husary_128kbps'),
 ('21', 'Husary_Mujawwad_64kbps'),
 ('22', 'Husary_128kbps_Mujawwad'),
 ('23', 'Hudhaify_32kbps'),
 ('24', 'Hudhaify_64kbps'),
 ('25', 'Hudhaify_128kbps'),
 ('26', 'Ibrahim_Akhdar_32kbps'),
 ('27', 'Ibrahim_Akhdar_64kbps'),
 ('28', 'Maher_AlMuaiqly_64

In [157]:
json.dump(recitations_json, open('recitationsCleaned.json', 'w'))

In [39]:
for _i, surah in tqdm(enumerate(info_json['data']['surahs'])):
    i = _i + 1
    Path(os.path.join(identifier, str(i))).mkdir(parents=True, exist_ok=True)
    for _j, ayah in enumerate(surah['ayahs']):
        j = _j + 1
        url = ayah['audio']
        r = requests.get(url, allow_redirects=True, verify=False)
        open(os.path.join(identifier, str(i), str(j)+'.mp3'), 'wb').write(r.content)

5it [08:06, 97.33s/it]


KeyboardInterrupt: 

In [25]:
surah

{'number': 1,
 'name': 'سُورَةُ ٱلْفَاتِحَةِ',
 'englishName': 'Al-Faatiha',
 'englishNameTranslation': 'The Opening',
 'revelationType': 'Meccan',
 'ayahs': [{'number': 1,
   'audio': 'https://cdn.islamic.network/quran/audio/128/ar.alafasy/1.mp3',
   'audioSecondary': ['https://cdn.islamic.network/quran/audio/128/ar.alafasy/1.mp3',
    'https://cdn.islamic.network/quran/audio/64/ar.alafasy/1.mp3'],
   'text': '\ufeffبِسْمِ ٱللَّهِ ٱلرَّحْمَٰنِ ٱلرَّحِيمِ',
   'numberInSurah': 1,
   'juz': 1,
   'manzil': 1,
   'page': 1,
   'ruku': 1,
   'hizbQuarter': 1,
   'sajda': False},
  {'number': 2,
   'audio': 'https://cdn.islamic.network/quran/audio/128/ar.alafasy/2.mp3',
   'audioSecondary': ['https://cdn.islamic.network/quran/audio/128/ar.alafasy/2.mp3',
    'https://cdn.islamic.network/quran/audio/64/ar.alafasy/2.mp3'],
   'text': 'ٱلْحَمْدُ لِلَّهِ رَبِّ ٱلْعَٰلَمِينَ',
   'numberInSurah': 2,
   'juz': 1,
   'manzil': 1,
   'page': 1,
   'ruku': 1,
   'hizbQuarter': 1,
   'sajda': False}

In [25]:
surahInfo = json.load(open('surahInfo.json', 'r'))

In [162]:
prefixes = ['aal', 'ad', 'adh', 'al', 'an', 'ar', 'as', 'ash', 'at', 'az']
def text_cleaned(text):
    _text = text
    for prefix in prefixes:
        if _text.startswith(prefix):
            if '-' in _text:
                return _text.replace(prefix+'-', '')
            #else:
            #    return _text.replace(prefix, '')
    else:
        return _text
names_cleaned = [text_cleaned(item["englishName"].lower()) for item in surahInfo['data']]

In [184]:
def func(guess):
    guess_lower = guess.lower()
    lst = []
    for i, item in enumerate(surahInfo['data']):
        name = item['englishName']
        name_lower = name.lower()
        name_cleaned = names_cleaned[i]
        if guess_lower in name_lower:
            lst.append((name, 0, name_lower))
        else:
            guess_cleaned = text_cleaned(guess_lower)
            #lst.append((name, editdistance.eval(name_lower, guess_lower)))
            lst.append((name, editdistance.eval(name_cleaned, guess_cleaned), name_cleaned))
    return sorted(lst, key = lambda x: (x[1], x[2]))

In [ ]:
func("alaa")

In [19]:
for item in tqdm(foo['chapters']):
    id_ = item["id"]
    verses_count = item["verses_count"]
    for verse_count in range(1, verses_count + 1):
        with urllib.request.urlopen("https://api.quranwbw.com/{}/{}".format(id_, verse_count)) as url:
            data = json.loads(url.read().decode())
            with open('Test/{}_{}.json'.format(id_, verse_count), 'w') as fp:
                json.dump(data, fp)

100%|██████████| 114/114 [15:36<00:00,  8.21s/it]


In [28]:
data.keys()

dict_keys(['ayahs', 'meta', 'words'])

In [63]:
import os
yep = {'word_arabic_uthmani', 'word_translation', 'word_transliteration'}
for i in tqdm(range(1, 114+1)): 
    for v in range(1, foo["chapters"][i-1]['verses_count'] + 1):
        data = json.load(open("Test/{}_{}.json".format(i, v), 'r'))
        data['ayahs'] = {'ayah_uthmani': data['ayahs']['ayah_uthmani']}
        del data['meta']
        data['words'] = [{key: value for key, value in word_dict.items() if key in yep} for word_dict in data['words']]
        json.dump(data, open('Test_cleaned/{}_{}.json'.format(i, v), 'w'))

100%|██████████| 114/114 [00:17<00:00,  6.69it/s]


In [64]:
data

{'ayahs': {'ayah_uthmani': 'مِنَ ٱلۡجِنَّةِ وَٱلنَّاسِ'},
 'words': [{'word_arabic_uthmani': 'مِنَ',
   'word_translation': 'From',
   'word_transliteration': 'mina'},
  {'word_arabic_uthmani': 'ٱلۡجِنَّةِ',
   'word_translation': 'the jinn',
   'word_transliteration': 'al-jinati'},
  {'word_arabic_uthmani': 'وَٱلنَّاسِ',
   'word_translation': 'and men',
   'word_transliteration': 'wal-nasi'}]}

In [47]:
data = json.load(open("Test/{}_{}.json".format(1, 1), 'r'))

In [54]:
json.dump(data, open('Test_cleaned/1_1.json', 'w'))

In [50]:
data['words'][0]

{'word_arabic': 'بِسْمِ',
 'word_arabic_indopak': 'بِسْمِ',
 'word_arabic_uthmani': 'بِسۡمِ',
 'word_ayah': 1,
 'word_corpus': {'forms': {'active_participle': 'مُسَمٍّ',
   'imperative': 'سَمِّ',
   'imperfect': 'يُسَمِّي',
   'lemma': 'اسْم',
   'passive_participle': 'مُسَمًّى',
   'perfect': 'سَمّٰى',
   'verbal_noun': 'تَسْمِيَةً'},
  'grammar': [{'text': 'بِ', 'type': 'preposition'},
   {'text': 'سْمِ', 'type': 'noun'}]},
 'word_number_in_ayah': 1,
 'word_number_in_quran': 1,
 'word_number_in_surah': 1,
 'word_root': 'س م و',
 'word_surah': 1,
 'word_translation': 'In (the) name',
 'word_transliteration': "bis'mi"}

In [98]:
import http.client
from tqdm import tqdm

In [102]:
conn = http.client.HTTPSConnection("api.quran.com")
payload = "{}"
url = "/api/v4/resources/translations/{}/info"
found = set()
for i in tqdm(range(0, 200)):
    conn.request("GET", url.format(i), payload)
    res = conn.getresponse()
    data = json.loads(res.read().strip())
    if 'error' not in data:
        print(i)
        found.add(i)

 10%|█         | 21/200 [00:05<00:52,  3.41it/s]19
20
 12%|█▏        | 24/200 [00:06<00:48,  3.62it/s]22
23
 14%|█▎        | 27/200 [00:07<00:41,  4.19it/s]25
26
 14%|█▍        | 29/200 [00:07<00:29,  5.74it/s]27
28
 16%|█▌        | 31/200 [00:07<00:24,  6.98it/s]29
30
 16%|█▋        | 33/200 [00:07<00:21,  7.83it/s]31
32
 18%|█▊        | 37/200 [00:08<00:31,  5.20it/s]35
36
 20%|█▉        | 39/200 [00:08<00:24,  6.59it/s]37
38
 22%|██▏       | 43/200 [00:09<00:29,  5.26it/s]41
42
 22%|██▎       | 45/200 [00:10<00:23,  6.64it/s]43
44
 24%|██▎       | 47/200 [00:10<00:20,  7.55it/s]45
46
 24%|██▍       | 49/200 [00:10<00:18,  8.16it/s]47
48
 26%|██▌       | 51/200 [00:10<00:18,  8.19it/s]49
50
 26%|██▋       | 53/200 [00:10<00:17,  8.33it/s]51
52
 28%|██▊       | 55/200 [00:11<00:16,  8.58it/s]53
54
 28%|██▊       | 57/200 [00:11<00:16,  8.69it/s]55
56
 38%|███▊      | 76/200 [00:17<00:23,  5.26it/s]74
75
 39%|███▉      | 78/200 [00:17<00:18,  6.54it/s]76
77
 40%|████      | 80/200 [00:

In [108]:
conn = http.client.HTTPSConnection("api.quran.com")
payload = "{}"
url = "/api/v4/resources/translations/{}/info"
for i in found:
    conn.request("GET", url.format(i), payload)
    res = conn.getresponse()
    data = json.loads(res.read().strip())
    print(data)

{'info': {'id': 17, 'info': None}}
{'info': {'id': 19, 'info': None}}
{'info': {'id': 20, 'info': None}}
{'info': {'id': 22, 'info': None}}
{'info': {'id': 23, 'info': None}}
{'info': {'id': 25, 'info': None}}
{'info': {'id': 26, 'info': None}}
{'info': {'id': 27, 'info': None}}
{'info': {'id': 28, 'info': None}}
{'info': {'id': 29, 'info': None}}
{'info': {'id': 30, 'info': None}}
{'info': {'id': 31, 'info': ''}}
{'info': {'id': 32, 'info': None}}
{'info': {'id': 33, 'info': None}}
{'info': {'id': 35, 'info': None}}
{'info': {'id': 36, 'info': None}}
{'info': {'id': 37, 'info': None}}
{'info': {'id': 38, 'info': None}}
{'info': {'id': 39, 'info': None}}
{'info': {'id': 41, 'info': None}}
{'info': {'id': 42, 'info': ''}}
{'info': {'id': 43, 'info': None}}
{'info': {'id': 44, 'info': ''}}
{'info': {'id': 45, 'info': None}}
{'info': {'id': 46, 'info': None}}
{'info': {'id': 47, 'info': None}}
{'info': {'id': 48, 'info': None}}
{'info': {'id': 49, 'info': None}}
{'info': {'id': 50, 'info'

In [1]:
import json

In [2]:
meta = json.load(open("moreSurahInfo.json", "r"))

In [7]:
surahInfo = json.load(open("surahInfo.json", "r"))

In [8]:
surahInfo

{'code': 200,
 'status': 'OK',
 'data': [{'number': 1,
   'name': 'سُورَةُ ٱلْفَاتِحَةِ',
   'englishName': 'Al-Faatiha',
   'englishNameTranslation': 'The Opening',
   'numberOfAyahs': 7,
   'revelationType': 'Meccan'},
  {'number': 2,
   'name': 'سُورَةُ البَقَرَةِ',
   'englishName': 'Al-Baqara',
   'englishNameTranslation': 'The Cow',
   'numberOfAyahs': 286,
   'revelationType': 'Medinan'},
  {'number': 3,
   'name': 'سُورَةُ آلِ عِمۡرَانَ',
   'englishName': 'Aal-i-Imraan',
   'englishNameTranslation': 'The Family of Imraan',
   'numberOfAyahs': 200,
   'revelationType': 'Medinan'},
  {'number': 4,
   'name': 'سُورَةُ النِّسَاءِ',
   'englishName': 'An-Nisaa',
   'englishNameTranslation': 'The Women',
   'numberOfAyahs': 176,
   'revelationType': 'Medinan'},
  {'number': 5,
   'name': 'سُورَةُ المَائـِدَةِ',
   'englishName': 'Al-Maaida',
   'englishNameTranslation': 'The Table',
   'numberOfAyahs': 120,
   'revelationType': 'Medinan'},
  {'number': 6,
   'name': 'سُورَةُ الأَنۡ

In [27]:
[(i+1, item) for i, item in enumerate(meta['data']['juzs']['references'])]

[(0, {'surah': 1, 'ayah': 1}),
 (1, {'surah': 2, 'ayah': 142}),
 (2, {'surah': 2, 'ayah': 253}),
 (3, {'surah': 3, 'ayah': 93}),
 (4, {'surah': 4, 'ayah': 24}),
 (5, {'surah': 4, 'ayah': 148}),
 (6, {'surah': 5, 'ayah': 82}),
 (7, {'surah': 6, 'ayah': 111}),
 (8, {'surah': 7, 'ayah': 88}),
 (9, {'surah': 8, 'ayah': 41}),
 (10, {'surah': 9, 'ayah': 93}),
 (11, {'surah': 11, 'ayah': 6}),
 (12, {'surah': 12, 'ayah': 53}),
 (13, {'surah': 15, 'ayah': 1}),
 (14, {'surah': 17, 'ayah': 1}),
 (15, {'surah': 18, 'ayah': 75}),
 (16, {'surah': 21, 'ayah': 1}),
 (17, {'surah': 23, 'ayah': 1}),
 (18, {'surah': 25, 'ayah': 21}),
 (19, {'surah': 27, 'ayah': 56}),
 (20, {'surah': 29, 'ayah': 46}),
 (21, {'surah': 33, 'ayah': 31}),
 (22, {'surah': 36, 'ayah': 28}),
 (23, {'surah': 39, 'ayah': 32}),
 (24, {'surah': 41, 'ayah': 47}),
 (25, {'surah': 46, 'ayah': 1}),
 (26, {'surah': 51, 'ayah': 31}),
 (27, {'surah': 58, 'ayah': 1}),
 (28, {'surah': 67, 'ayah': 1}),
 (29, {'surah': 78, 'ayah': 1})]

In [1]:
import os
import json
from tqdm import tqdm

In [7]:
quran_uthmani = json.load(open("quran-uthmani.json"))

In [31]:
for i, surah in tqdm(enumerate(quran_uthmani['data']['surahs'])):
    surah_number = i+1
    for j, ayah in enumerate(surah["ayahs"]):
        ayah_number = j+1
        fileinpath = f"/Users/pickle/Downloads/prayer/prayer/Surah_Ayah_og/{surah_number}_{ayah_number}.json"
        fileoutpath = f"/Users/pickle/Downloads/prayer/prayer/Surah_Ayah_out/{surah_number}_{ayah_number}.json"
        item = json.load(open(fileinpath, 'r'))
        item['juz'] = ayah['juz']
        item['manzil'] = ayah['manzil']
        item['page'] = ayah['page']
        item['ruku'] = ayah['ruku']
        item['hizbQuarter'] = ayah['hizbQuarter']
        item['sajda'] = ayah['sajda']
        #os.remove(fileoutpath)
        with open(fileoutpath, 'w') as file:
            json.dump(item, file)

114it [00:08, 13.92it/s]


In [25]:
str(item)

'{\'ayahs\': {\'ayah_uthmani\': \'بِسۡمِ ٱللَّهِ ٱلرَّحۡمَٰنِ ٱلرَّحِيمِ\'}, \'words\': [{\'word_arabic_uthmani\': \'بِسۡمِ\', \'word_translation\': \'In (the) name\', \'word_transliteration\': "bis\'mi"}, {\'word_arabic_uthmani\': \'ٱللَّهِ\', \'word_translation\': \'(of) Allah\', \'word_transliteration\': \'al-lahi\'}, {\'word_arabic_uthmani\': \'ٱلرَّحۡمَٰنِ\', \'word_translation\': \'the Most Gracious\', \'word_transliteration\': \'al-rahmani\'}, {\'word_arabic_uthmani\': \'ٱلرَّحِيمِ\', \'word_translation\': \'the Most Merciful\', \'word_transliteration\': \'al-rahimi\'}], \'juz\': 1, \'manzil\': 1, \'page\': 1, \'ruku\': 1, \'hizbQuarter\': 1, \'sajda\': False}'

In [20]:
item

{'ayahs': {'ayah_uthmani': 'مِنَ ٱلۡجِنَّةِ وَٱلنَّاسِ'},
 'words': [{'word_arabic_uthmani': 'مِنَ',
   'word_translation': 'From',
   'word_transliteration': 'mina'},
  {'word_arabic_uthmani': 'ٱلۡجِنَّةِ',
   'word_translation': 'the jinn',
   'word_transliteration': 'al-jinati'},
  {'word_arabic_uthmani': 'وَٱلنَّاسِ',
   'word_translation': 'and men',
   'word_transliteration': 'wal-nasi'}],
 'juz': 30,
 'manzil': 7,
 'page': 604,
 'ruku': 556,
 'hizbQuarter': 240,
 'sajda': False}

In [18]:
ayah

{'number': 6236,
 'text': 'مِنَ ٱلْجِنَّةِ وَٱلنَّاسِ',
 'numberInSurah': 6,
 'juz': 30,
 'manzil': 7,
 'page': 604,
 'ruku': 556,
 'hizbQuarter': 240,
 'sajda': False}

In [17]:
item

{'ayahs': {'ayah_uthmani': 'مِنَ ٱلۡجِنَّةِ وَٱلنَّاسِ'},
 'words': [{'word_arabic_uthmani': 'مِنَ',
   'word_translation': 'From',
   'word_transliteration': 'mina'},
  {'word_arabic_uthmani': 'ٱلۡجِنَّةِ',
   'word_translation': 'the jinn',
   'word_transliteration': 'al-jinati'},
  {'word_arabic_uthmani': 'وَٱلنَّاسِ',
   'word_translation': 'and men',
   'word_transliteration': 'wal-nasi'}]}

In [19]:
paths = []
for item in os.listdir("Surah_Ayah"):
    if not item.endswith("json"):
        continue
    path = f'Surah_Ayah/{item}'
    item = json.load(open(path, 'r'))
    if item['ayahs']['ayah_uthmani'].strip("\u06de").strip() != item['ayahs']['ayah_uthmani']:
        paths.append(path)
        #item['ayahs']['ayah_uthmani'] = item['ayahs']['ayah_uthmani'].strip("\u06de").strip()
        #with open(path, 'w') as file:
        #    json.dump(item, file)
print(sorted(paths))

['Surah_Ayah/100_9.json', 'Surah_Ayah/10_11.json', 'Surah_Ayah/10_26.json', 'Surah_Ayah/10_53.json', 'Surah_Ayah/10_71.json', 'Surah_Ayah/10_90.json', 'Surah_Ayah/11_108.json', 'Surah_Ayah/11_24.json', 'Surah_Ayah/11_41.json', 'Surah_Ayah/11_6.json', 'Surah_Ayah/11_61.json', 'Surah_Ayah/11_84.json', 'Surah_Ayah/12_101.json', 'Surah_Ayah/12_30.json', 'Surah_Ayah/12_53.json', 'Surah_Ayah/12_7.json', 'Surah_Ayah/12_77.json', 'Surah_Ayah/13_19.json', 'Surah_Ayah/13_35.json', 'Surah_Ayah/13_5.json', 'Surah_Ayah/14_10.json', 'Surah_Ayah/14_28.json', 'Surah_Ayah/15_49.json', 'Surah_Ayah/16_111.json', 'Surah_Ayah/16_30.json', 'Surah_Ayah/16_51.json', 'Surah_Ayah/16_75.json', 'Surah_Ayah/16_90.json', 'Surah_Ayah/17_23.json', 'Surah_Ayah/17_50.json', 'Surah_Ayah/17_70.json', 'Surah_Ayah/17_99.json', 'Surah_Ayah/18_17.json', 'Surah_Ayah/18_32.json', 'Surah_Ayah/18_51.json', 'Surah_Ayah/18_75.json', 'Surah_Ayah/18_99.json', 'Surah_Ayah/19_22.json', 'Surah_Ayah/19_59.json', 'Surah_Ayah/20_111.json'